In [ ]:
"""
models.py
---------
Tres arquitecturas de Deep Learning para el pipeline de Threat Hunting
Autonomo, alineadas al titulo de tesis doctoral (deteccion de amenazas
para optimizar playbooks de respuesta a incidentes en un SOC):

1. MLPClassifier        -> Red densa supervisada (baseline DL tabular).
2. LSTMClassifier        -> Red recurrente supervisada; el vector de
                             features de cada flujo se trata como una
                             pseudo-secuencia (cada feature = un timestep),
                             tecnica habitual en literatura de IDS con DL
                             para reutilizar arquitecturas secuenciales
                             sobre datos tabulares de flujo.
3. Autoencoder            -> Modelo no supervisado de deteccion de
                             anomalias. Se entrena solo con el patron de
                             trafico dominante del set de entrenamiento
                             (ver train_pipeline.py) y usa el error de
                             reconstruccion como score de anomalia.
"""

import torch
import torch.nn as nn


class MLPClassifier(nn.Module):
    def __init__(self, n_features: int, hidden_sizes=(128, 64, 32), dropout: float = 0.3):
        super().__init__()
        layers = []
        in_dim = n_features
        for h in hidden_sizes:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(dropout)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))  # logit binario
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class LSTMClassifier(nn.Module):
    def __init__(self, n_features: int, hidden_size: int = 32, num_layers: int = 1, dropout: float = 0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=1, hidden_size=hidden_size, num_layers=num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=False,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, n_features) -> (batch, n_features, 1) pseudo-secuencia
        x = x.unsqueeze(-1)
        out, (h_n, _) = self.lstm(x)
        h_last = self.dropout(h_n[-1])
        return self.fc(h_last).squeeze(-1)


class AutoencoderAnomaly(nn.Module):
    def __init__(self, n_features: int, latent_dim: int = 16, dropout: float = 0.1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 48), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(48, 24), nn.ReLU(),
            nn.Linear(24, latent_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 24), nn.ReLU(),
            nn.Linear(24, 48), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(48, n_features),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


def build_model(name: str, n_features: int):
    if name == "MLP":
        return MLPClassifier(n_features)
    if name == "LSTM":
        return LSTMClassifier(n_features)
    if name == "Autoencoder":
        return AutoencoderAnomaly(n_features)
    raise ValueError(f"Modelo desconocido: {name}")
